# Week 10 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [ ]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 10 Day 01: Volatility stylized facts

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Understand volatility dynamics and regime-change handling for risk-aware strategies.

## Continuity and Handoff
- Previous checkpoint: Week 09 Day 07: Portfolio Project
- Previous lesson file: content/week-09/day-07.md
- Today's deliverable: Compute and plot multiple volatility estimators.
- Next handoff target: Week 10 Day 02: EWMA volatility estimation
- Next lesson file: content/week-10/day-02.md

## Theory Concepts

### Concept 1: Volatility clustering
Volatility clustering should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Leverage effect intuition
Leverage effect intuition should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Persistence of shocks
Persistence of shocks should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
Removes non-stationary level drift.

### Formula 2: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
Lag-memory measurement.

### Formula 3: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
One-step dependence model.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\phi$: autoregressive coefficient
- $e_t$: forecast residual
- $\lambda$: EWMA decay

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Visualize clustered volatility periods in rolling windows.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Volatility stylized facts'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Compute and plot multiple volatility estimators.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Why do volatility shocks persist longer than return shocks?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1001)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


# Week 10 Day 02: EWMA volatility estimation

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Understand volatility dynamics and regime-change handling for risk-aware strategies.

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 01: Volatility stylized facts
- Previous lesson file: content/week-10/day-01.md
- Today's deliverable: Implement EWMA estimator with configurable lambda.
- Next handoff target: Week 10 Day 03: GARCH intuition
- Next lesson file: content/week-10/day-03.md

## Theory Concepts

### Concept 1: Exponential weighting
Exponential weighting should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Decay parameter interpretation
Decay parameter interpretation should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Responsiveness vs smoothness tradeoff
Responsiveness vs smoothness tradeoff should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
Lag-memory measurement.

### Formula 2: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
One-step dependence model.

### Formula 3: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
Adaptive volatility estimate.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\phi$: autoregressive coefficient
- $e_t$: forecast residual
- $\lambda$: EWMA decay

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Compare EWMA volatility under fast and slow decay.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'EWMA volatility estimation'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement EWMA estimator with configurable lambda.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How should lambda change across asset classes?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1002)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


# Week 10 Day 03: GARCH intuition

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Understand volatility dynamics and regime-change handling for risk-aware strategies.

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 02: EWMA volatility estimation
- Previous lesson file: content/week-10/day-02.md
- Today's deliverable: Estimate GARCH and compare conditional volatility path.
- Next handoff target: Week 10 Day 04: Regime detection and change points
- Next lesson file: content/week-10/day-04.md

## Theory Concepts

### Concept 1: Conditional heteroskedasticity
Conditional heteroskedasticity should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: ARCH/GARCH terms
ARCH/GARCH terms should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Parameter stability
Parameter stability should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: AR(1)
$$
x_t=c+\phi x_{t-1}+\epsilon_t
$$
One-step dependence model.

### Formula 2: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
Adaptive volatility estimate.

### Formula 3: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
Forecast error benchmark.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\phi$: autoregressive coefficient
- $e_t$: forecast residual
- $\lambda$: EWMA decay

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Fit a simple GARCH model on synthetic clustered volatility.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'GARCH intuition'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Estimate GARCH and compare conditional volatility path.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
What does unstable GARCH calibration signal about regime change?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1003)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


# Week 10 Day 04: Regime detection and change points

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Understand volatility dynamics and regime-change handling for risk-aware strategies.

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 03: GARCH intuition
- Previous lesson file: content/week-10/day-03.md
- Today's deliverable: Implement basic change-point detection and regime labels.
- Next handoff target: Week 10 Day 05: Volatility-aware decisioning
- Next lesson file: content/week-10/day-05.md

## Theory Concepts

### Concept 1: Structural breaks
Structural breaks should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Change-point algorithms
Change-point algorithms should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Regime labeling workflows
Regime labeling workflows should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: EWMA Vol
$$
\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2
$$
Adaptive volatility estimate.

### Formula 2: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
Forecast error benchmark.

### Formula 3: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
Removes non-stationary level drift.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\phi$: autoregressive coefficient
- $e_t$: forecast residual
- $\lambda$: EWMA decay

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Detect breakpoints in a volatility series with synthetic shifts.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Regime detection and change points'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement basic change-point detection and regime labels.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How should strategy exposure adapt at detected breakpoints?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1004)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


# Week 10 Day 05: Volatility-aware decisioning

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Understand volatility dynamics and regime-change handling for risk-aware strategies.

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 04: Regime detection and change points
- Previous lesson file: content/week-10/day-04.md
- Today's deliverable: Build a simple volatility-targeting overlay.
- Next handoff target: Week 10 Day 06: Revision Sprint
- Next lesson file: content/week-10/day-06.md

## Theory Concepts

### Concept 1: Volatility targeting
Volatility targeting should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Adaptive position sizing
Adaptive position sizing should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Risk budget adjustments
Risk budget adjustments should be treated as a measurable component of 'Volatility modeling and regime shifts'. For this week, emphasize temporal dependence structure and out-of-sample forecast discipline. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: RMSE
$$
RMSE=\sqrt{\frac{1}{n}\sum_t e_t^2}
$$
Forecast error benchmark.

### Formula 2: First Difference
$$
\Delta x_t=x_t-x_{t-1}
$$
Removes non-stationary level drift.

### Formula 3: Autocorrelation
$$
\rho_k=\frac{Cov(x_t,x_{t-k})}{Var(x_t)}
$$
Lag-memory measurement.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\phi$: autoregressive coefficient
- $e_t$: forecast residual
- $\lambda$: EWMA decay

## Real Trading Example
- Instruments: SPY, TLT, GLD
- Macro overlay (FRED): VIXCLS, DGS2
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Simulate position scaling under different volatility regimes.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Volatility-aware decisioning'.

## Step-by-Step Solved Problems
### Solved Problem 1: One-step AR(1) forecast
Given:
- Use c=0.001, phi=0.64, x_t=0.015.
Solution:
1. $x_{t+1}=c+\phi x_t$.
2. Forecast = 0.001 + 0.64*0.015 = 0.010600.
Final answer: Forecasted value = 1.06%.

### Solved Problem 2: EWMA volatility update
Given:
- lambda=0.94, sigma_(t-1)=0.020, r_(t-1)=-0.012.
Solution:
1. $\sigma_t^2=\lambda\sigma_{t-1}^2+(1-\lambda)r_{t-1}^2$.
2. sigma_t^2 = 0.94*(0.020^2) + 0.06*(0.012^2) = 0.00038464.
3. sigma_t = sqrt(0.00038464) = 0.01961.
Final answer: Updated volatility = 1.96%.

### Solved Problem 3: Compute RMSE
Given:
- Errors are [0.004, -0.006, 0.003, -0.002, 0.005].
Solution:
1. $RMSE=\sqrt{\frac{1}{n}\sum e_t^2}$.
2. Mean squared error = 0.000018.
3. RMSE = sqrt(0.000018) = 0.00424.
Final answer: RMSE = 0.424%.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build a simple volatility-targeting overlay.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
series = build_stationary_series(price_df['Close'])
forecast = walk_forward_arima(series, order=(1, 1, 1))
errors = series.loc[forecast.index] - forecast
rmse = np.sqrt(np.mean(errors**2))
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
When can volatility targeting increase hidden risks?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# Time-series demo: AR-style diagnostics on real returns
np.random.seed(1005)
series = load_market_prices(['SPY'], start='2010-01-01')['SPY'].pct_change().dropna()
x_prev = series.shift(1).dropna()
x_curr = series.loc[x_prev.index]

phi = float(np.cov(x_curr.values, x_prev.values, ddof=1)[0, 1] / np.var(x_prev.values, ddof=1))
forecast_next = float(phi * series.iloc[-1])
acf1 = float(series.autocorr(lag=1))
acf5 = float(series.autocorr(lag=5))
rmse_naive = float(np.sqrt(np.mean((series.iloc[1:].values - series.shift(1).dropna().values) ** 2)))

{
    'ar1_phi': phi,
    'next_return_forecast': forecast_next,
    'acf_lag1': acf1,
    'acf_lag5': acf5,
    'naive_rmse': rmse_naive,
}


# Week 10 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 05: Volatility-aware decisioning
- Previous lesson file: content/week-10/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 10 Day 07: Portfolio Project
- Next lesson file: content/week-10/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Recreate EWMA and GARCH estimates from memory
- Compare change-point outputs under different thresholds
- Update risk-budget notes for regime transitions

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 10 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 10 Day 06: Revision Sprint
- Previous lesson file: content/week-10/day-06.md
- Today's deliverable: Volatility regime labeling project
- Next handoff target: Week 11 Day 01: Backtest design patterns
- Next lesson file: content/week-11/day-01.md

## Project Title
Volatility regime labeling project

## Problem Statement
Build a volatility regime engine and connect labels to risk controls.

## Data Sources
- Asset returns
- Rolling vol estimates
- Synthetic regime labels

## Implementation Steps
1. Estimate multiple volatility measures
2. Detect potential regime shifts
3. Create regime labels
4. Map labels to risk controls
5. Summarize governance recommendations

## Evaluation Metrics
- Regime stability
- Detection lag
- Risk-control fit
- Interpretability

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
